# NorthStar — Day 8: forecasts and causal effects

Runs **Chronos-2** (zero-shot probabilistic forecasts) and **DoubleML** (effect of Bank of Canada rate changes on sector returns) on the data your daily pipeline already commits to GitHub. Outputs two files for the website: `forecasts.json` and `causal_effects.json`.

**Runtime:** a GPU (Runtime → Change runtime type → T4) makes Chronos-2 faster, but CPU works too. Total time is roughly 5–15 minutes.

In [ ]:
!git clone -q https://github.com/jeduclo/northstar.git
%cd northstar
!git log -1 --format='Data as of commit %h (%cd)' --date=short

In [ ]:
!pip install -q "chronos-forecasting>=2.0" doubleml

## 1. Forecasts (Chronos-2)
12-month P10/P50/P90 paths for key macro series, a 12-month holdout backtest against a naive forecast, and 12-month return ranges for every sector ETF.

In [ ]:
!python -m pipeline.models.forecast

## 2. Causal effects (DoubleML)
Effect of a +0.25 pp Bank of Canada rate change on same-month sector returns, controlling for U.S. inflation and rates, oil, the Canadian dollar and labour markets. Set `NORTHSTAR_DML_REPS` higher (e.g. 10) for more stable estimates.

In [ ]:
import os
os.environ["NORTHSTAR_DML_REPS"] = "5"
!python -m pipeline.models.causal

## 3. Quick look

In [ ]:
import json, pandas as pd
fc = json.load(open("web/public/data/forecasts.json"))
print(pd.DataFrame([{"series": m["name"], "P50 in 12m": m["forecast"][-1][2],
                    "MAE Chronos": m["backtest"]["mae_model"], "MAE naive": m["backtest"]["mae_naive"]} for m in fc["macro"]]).to_string(index=False))
ce = json.load(open("web/public/data/causal_effects.json"))
print("\nPolicy moves in sample:", ce["policy_moves"])
pd.DataFrame(ce["effects"])[["market", "sector", "coef", "ci_low", "ci_high", "p_value", "n_obs"]]

## 4. Download the results
Unzip into your repo (`web/public/data/`), then commit and push from your PC.

In [ ]:
!cd web/public/data && zip -q ../../../model_outputs.zip forecasts.json causal_effects.json
from google.colab import files
files.download("model_outputs.zip")